# 01 · Thu thập dữ liệu VSL

Nhập **danh sách lớp + số mẫu** một lần (vd. `a:20; b:20; xin chào:15`, phân tách bằng `;` vì tên lớp có thể chứa dấu phẩy như `dấu â,ô,ê`), chạy ô thu thập, rồi đứng trước camera. Một cửa sổ OpenCV sẽ hiện ra.

### Chế độ `MODE = "auto"` (mặc định): rảnh tay
Mỗi mẫu tự chạy theo vòng: **hạ tay về tư thế nghỉ** (dưới bụng hoặc ra khỏi khung) ~`ARM_SECONDS` giây → *tích* đếm ngược → *bụp* bắt đầu quay `RECORD_SECONDS` giây → tự kiểm tra chất lượng → **tự lưu** (*ting*) hoặc **tự bỏ** (*bùm*, kèm lý do) → lặp lại. Hết số mẫu của một lớp thì tự sang lớp kế tiếp. Viền cửa sổ đổi màu theo trạng thái để nhìn từ xa: xanh lá = chờ hạ tay, vàng = đếm ngược, đỏ = đang quay, xám = tạm dừng.

Không cần chạm bàn phím, trừ khi muốn:

| Phím | Tác dụng |
|---|---|
| `SPACE` | tạm dừng / tiếp tục (đang quay thì huỷ mẫu đó) |
| `U` | xoá mẫu vừa lưu gần nhất |
| `N` | sang lớp kế tiếp |
| `Q` | thoát |

Không có người trong khung thì không quay. Bị bỏ `MAX_REJECTS` mẫu liên tiếp thì tự tạm dừng. Nếu tay nghỉ của bạn vẫn nằm trong vùng "đang ký" (vd. ngồi tì tay lên bàn) nên vòng không tự bắt đầu, đặt `ARM_MODE = "timer"` để quay cách nhau `GAP_SECONDS` giây.

### Chế độ `MODE = "manual"`: duyệt từng mẫu như cũ
`SPACE` bắt đầu · `ENTER` lưu (khi đạt chất lượng) · `F` lưu dù chất lượng thấp · `R` bỏ mẫu vừa quay · `U` xoá mẫu vừa lưu · `N` lớp kế · `Q` thoát.

Thay vì duyệt từng mẫu, có thể quay auto rồi xem lại cả loạt ở phần **Xem lại** cuối notebook.

Chạy bằng Jupyter **cục bộ** trên máy có webcam (cửa sổ OpenCV không hiện trên Colab/remote). Âm báo dùng `afplay` của macOS; máy khác thì im lặng.
Mỗi mẫu lưu **đặc trưng thô theo từng frame** (có thể chứa NaN khi mất tay) + video gốc + `meta.json`.

In [1]:
import subprocess, sys, time, unicodedata
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.builtin.mediapipe_builtin import LandmarkerLoader, VisionRunningMode
from src.builtin.mediapipe_builtin.defs import PoseLandmarkerPath, HandLandmarkerPath
from src.data.data_reader import DataHandler
from src.features.extract import extract, active_presence, FEATURE_DIM, FEATURE_VERSION
from src.features.camera import open_camera
from src.features.frame import FRAME_SIZE, frame_padding

MODE = "auto"
DEFAULT_SAMPLES = 20

CLASS_PLAN = {}

def parse_plan(text, default=DEFAULT_SAMPLES):
    plan = {}
    for item in text.split(";"):
        name, _, n = item.rpartition(":") if ":" in item else (item, "", "")
        name = name.strip()
        if not name:
            continue
        n = int(n) if n.strip() else default
        if n < 1:
            raise ValueError(f"Số mẫu của '{name}' phải >= 1")
        plan[unicodedata.normalize("NFC", name)] = n
    if not plan:
        raise ValueError("Chưa nhập lớp nào")
    return plan

if not CLASS_PLAN:
    CLASS_PLAN = parse_plan(input(f"Lớp và số mẫu (vd. a:20; b:20; xin chào; mặc định {DEFAULT_SAMPLES}): "))
print("Kế hoạch:", ", ".join(f"{k}: {v}" for k, v in CLASS_PLAN.items()))

CAMERA_INDEX = 0
CAMERA_RESOLUTION = None
MIRROR_DATA = False
MIRROR_PREVIEW = True
COUNTDOWN_SECONDS = 1.0
RECORD_SECONDS = 2.0
MIN_POSE_COVERAGE = 0.8
MIN_ACTIVE_RATIO = 0.25

ARM_MODE = "rest"
ARM_SECONDS = 0.5
GAP_SECONDS = 1.5
INTRO_SECONDS = 3.0
MAX_REJECTS = 5
SOUND = True

DISPLAY_SIZE = 720
WINDOW = "VSL collect"

POSE_MODEL_PATH = Path(PoseLandmarkerPath)
HAND_MODEL_PATH = Path(HandLandmarkerPath)
DETECTION_CONF = dict(detection=0.5, presence=0.5, tracking=0.5)

assert MODE in ("auto", "manual") and ARM_MODE in ("rest", "timer")
SESSION = datetime.now().strftime("%Y%m%d-%H%M%S")
print("Session:", SESSION, "| mode:", MODE)

Kế hoạch: gặp: 20
Session: 20261126-230129 | mode: auto


In [2]:
handler = DataHandler()

assert FEATURE_DIM == DataHandler.FEATURE_DIM, (
    f"Extractor trả {FEATURE_DIM} chiều nhưng schema.FRAME_FEATURE_DIM = {DataHandler.FEATURE_DIM}. "
    "Cập nhật FRAME_FEATURE_DIM trong src/features/schema.py."
)
for p in (POSE_MODEL_PATH, HAND_MODEL_PATH):
    assert p.is_file(), f"Không thấy file model: {p}"

def print_table():
    counts = handler.get_class_counts()
    print(f"{'Lớp':<28}{'Số mẫu':>8}")
    print("-" * 36)
    for name, n in counts.items():
        print(f"{name:<28}{n:>8}")
    print("-" * 36)
    print(f"{len(counts)} lớp, {sum(counts.values())} mẫu")
    if handler.incomplete:
        print("\n[cảnh báo] mẫu dở dang/hỏng bị bỏ qua:")
        for path, why in handler.incomplete:
            print("  ", path, "->", why)

print_table()

Lớp                           Số mẫu
------------------------------------
a                                 45
age                               45
b                                 45
c                                 45
cảm ơn                            45
dấu huyền                         45
dấu hỏi                           45
dấu nặng                          45
dấu sắc                           45
dấu â,ô,ê                         45
dấu ă                             45
dấu ơ,ư                           45
e                                 45
g                                 45
h                                 45
i                                 45
k                                 45
l                                 45
m                                 45
n                                 45
o                                 45
r                                 45
t                                 45
ten_la                            45
toi                               45
u

In [3]:
class Detector:
    """Pose + Hand landmarker (chế độ VIDEO, timestamp luôn tăng) -> đặc trưng 1 frame."""

    def __init__(self):
        c = DETECTION_CONF
        self.landmarkers = LandmarkerLoader(
            running_mode=VisionRunningMode.VIDEO,
            pose_model_path=POSE_MODEL_PATH, hand_model_path=HAND_MODEL_PATH,
            detection_confidence=c["detection"], presence_confidence=c["presence"],
            tracking_confidence=c["tracking"])

    def __call__(self, frame_bgr):
        pose_res, hand_res, _ = self.landmarkers.detect(frame_bgr)
        return extract(pose_res, hand_res), pose_res, hand_res

    def close(self):
        self.landmarkers.close()

_SOUNDS = {"tick": "Tink", "go": "Pop", "saved": "Glass", "bad": "Basso"}

def beep(kind):
    """Phát âm báo không chặn vòng lặp (chỉ macOS; máy khác bỏ qua)."""
    if not SOUND or sys.platform != "darwin":
        return
    try:
        subprocess.Popen(["afplay", f"/System/Library/Sounds/{_SOUNDS[kind]}.aiff"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except OSError:
        pass

def ascii_text(s):
    """cv2.putText không vẽ được chữ có dấu -> bỏ dấu khi hiển thị trên cửa sổ."""
    s = s.replace("đ", "d").replace("Đ", "D")
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

def put(img, text, y, color=(255, 255, 255), scale=0.65, center=False):
    thick = max(2, int(scale * 2))
    x = 14
    if center:
        (w, _), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
        x = max(14, (img.shape[1] - w) // 2)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thick + 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick, cv2.LINE_AA)

POSE_DRAW = [0, 11, 12, 13, 14, 15, 16]

def draw_overlay(img, pose_res, hand_res):
    h, w = img.shape[:2]
    if pose_res and pose_res.pose_landmarks:
        for i in POSE_DRAW:
            lm = pose_res.pose_landmarks[0][i]
            cv2.circle(img, (int(lm.x * w), int(lm.y * h)), 6, (0, 200, 255), -1)
    if hand_res and hand_res.hand_landmarks:
        for lms in hand_res.hand_landmarks:
            for lm in lms:
                cv2.circle(img, (int(lm.x * w), int(lm.y * h)), 3, (0, 255, 0), -1)

def draw_border(img, color, thickness=14):
    h, w = img.shape[:2]
    cv2.rectangle(img, (0, 0), (w - 1, h - 1), color, thickness)

def summarize(feats):
    """(tỉ lệ frame có pose, có tay trái, có tay phải, có tay giơ lên khỏi vị trí nghỉ)"""
    arr = np.stack(feats)
    pres = np.nan_to_num(arr[:, DataHandler.PRESENCE_COLS])
    active = np.array([active_presence(f).any() for f in feats])
    return (float((~np.isnan(arr).all(axis=1)).mean()), float(pres[:, 0].mean()),
            float(pres[:, 1].mean()), float(active.mean()))

def quality_issue(feats, summary):
    """None nếu mẫu đạt, ngược lại là lý do (tiếng Việt) không đạt."""
    if len(feats) < DataHandler.MIN_FRAMES:
        return f"quá ít frame ({len(feats)} < {DataHandler.MIN_FRAMES})"
    if summary[0] < MIN_POSE_COVERAGE:
        return f"mất pose ({summary[0]:.0%} frame có pose)"
    if summary[1] == 0 and summary[2] == 0:
        return "không thấy tay"
    if summary[3] < MIN_ACTIVE_RATIO:
        return f"tay chưa giơ lên ({summary[3]:.0%} frame)"
    return None

In [4]:
KEY_SPACE, KEY_ENTER = 32, (13, 10)
COLORS = {"READY": (0, 200, 0), "ARMING": (0, 200, 0), "INTRO": (255, 200, 0), "COUNTDOWN": (0, 220, 255),
          "RECORDING": (0, 0, 255), "REVIEW": (255, 255, 255), "PAUSED": (128, 128, 128)}

def collect_class(cap, det, class_name, n_samples, mode=MODE):
    """Thu thập `n_samples` mẫu MỚI cho 1 lớp. Trả về "done" | "next" | "quit"."""
    auto = mode == "auto"
    name = handler.add_class(class_name)
    label = ascii_text(name)
    saved = []
    total = handler.get_class_counts().get(name, 0)
    state, t0 = ("INTRO" if auto else "READY"), time.time()
    rest_since, rejects = None, 0
    feats, frames, summary, issue, fps_est, msg = [], [], None, None, 30.0, ""

    def log(text):
        nonlocal msg
        msg = ascii_text(text)
        print(text)

    def go(new_state):
        nonlocal state, t0, rest_since
        state, t0, rest_since = new_state, time.time(), None

    def save():
        nonlocal total
        try:
            idx = handler.write_sample(
                name, feats, frames, fps=fps_est, session=SESSION,
                meta={"mirror": MIRROR_DATA, "record_seconds": RECORD_SECONDS,
                      "feature_version": FEATURE_VERSION, "collect_mode": mode,
                      "pose_coverage": summary[0], "left_hand": summary[1], "right_hand": summary[2]})
        except ValueError as e:
            log(f"Không lưu được: {e}")
            beep("bad")
            return False
        saved.append(idx)
        total = handler.get_class_counts().get(name, 0)
        log(f"Đã lưu {name}/{idx} ({len(saved)}/{n_samples}, {len(feats)} frame, {fps_est:.1f} fps)")
        beep("saved")
        return True

    while len(saved) < n_samples:
        ok, raw = cap.read()
        if not ok:
            raise RuntimeError("Không đọc được frame từ camera")
        if MIRROR_DATA:
            raw = cv2.flip(raw, 1)
        frame = frame_padding(raw)
        if frame.shape != (FRAME_SIZE, FRAME_SIZE, 3):
            raise RuntimeError(f"frame_padding phải trả ảnh {FRAME_SIZE}x{FRAME_SIZE}, nhận shape {frame.shape}")
        feat, pose_res, hand_res = det(frame)
        has_pose = bool(pose_res and pose_res.pose_landmarks)
        now = time.time()

        if state == "INTRO" and now - t0 >= INTRO_SECONDS:
            go("ARMING")
        if state == "ARMING":
            if ARM_MODE == "timer":
                armed = has_pose and now - t0 >= GAP_SECONDS
            else:
                resting = has_pose and not active_presence(feat).any()
                rest_since = (rest_since or now) if resting else None
                armed = rest_since is not None and now - rest_since >= ARM_SECONDS
            if armed:
                go("COUNTDOWN")
                beep("tick")
        if state == "COUNTDOWN" and now - t0 >= COUNTDOWN_SECONDS:
            go("RECORDING")
            feats, frames = [], []
            beep("go")
        if state == "RECORDING":
            feats.append(feat)
            frames.append(frame.copy())
            if now - t0 >= RECORD_SECONDS:
                fps_est = len(feats) / max(now - t0, 1e-6)
                summary = summarize(feats)
                issue = quality_issue(feats, summary)
                if not auto:
                    go("REVIEW")
                elif issue is None:
                    rejects = 0 if save() else rejects + 1
                    go("ARMING")
                else:
                    rejects += 1
                    log(f"Bỏ mẫu: {issue}")
                    beep("bad")
                    go("ARMING")
                if auto and rejects >= MAX_REJECTS:
                    log(f"Bỏ {rejects} mẫu liên tiếp -> tạm dừng. Kiểm tra khung hình, SPACE để tiếp tục")
                    go("PAUSED")
                if len(saved) >= n_samples:
                    break

        disp = cv2.resize(frame, (DISPLAY_SIZE, DISPLAY_SIZE), interpolation=cv2.INTER_AREA)
        draw_overlay(disp, pose_res, hand_res)
        if MIRROR_PREVIEW:
            disp = cv2.flip(disp, 1)
        draw_border(disp, COLORS[state])
        mid = DISPLAY_SIZE // 2
        put(disp, f"{label}  [{len(saved)}/{n_samples} moi | tong {total}]", 36, (255, 255, 0), 0.8)
        if state == "INTRO":
            put(disp, label, mid, (255, 255, 0), 2.2, center=True)
            put(disp, f"bat dau sau {max(0.0, INTRO_SECONDS - (now - t0)):.0f}s", mid + 50, center=True)
        elif state == "ARMING":
            if not has_pose:
                put(disp, "Khong thay nguoi trong khung", mid, (0, 200, 255), 1.0, center=True)
            elif ARM_MODE == "timer":
                put(disp, f"Mau tiep sau {max(0.0, GAP_SECONDS - (now - t0)):.1f}s", mid, (0, 255, 0), 1.0, center=True)
            else:
                put(disp, "Ha tay xuong de quay mau tiep", mid, (0, 255, 0), 1.0, center=True)
                if rest_since is not None:
                    frac = min(1.0, (now - rest_since) / ARM_SECONDS)
                    cv2.rectangle(disp, (mid - 150, mid + 25), (mid - 150 + int(300 * frac), mid + 40), (0, 255, 0), -1)
                elif now - t0 > 6:
                    put(disp, "Tay nghi phai thap hon bung (hoac ARM_MODE='timer')", mid + 50, (0, 200, 255), 0.6,
                        center=True)
        elif state == "READY":
            put(disp, "SPACE: bat dau | U: xoa mau vua luu | N: lop ke | Q: thoat", 70)
        elif state == "COUNTDOWN":
            put(disp, f"{max(0.0, COUNTDOWN_SECONDS - (now - t0)):.1f}", mid, (0, 220, 255), 3.0, center=True)
        elif state == "RECORDING":
            cv2.circle(disp, (DISPLAY_SIZE - 40, 80), 14, (0, 0, 255), -1)
            put(disp, f"DANG QUAY {now - t0:.1f}/{RECORD_SECONDS:.1f}s  ({len(feats)} frame)", 70, (0, 0, 255))
        elif state == "PAUSED":
            put(disp, "TAM DUNG", mid, (200, 200, 200), 2.0, center=True)
            put(disp, "SPACE: tiep tuc", mid + 50, center=True)
        else:
            put(disp, f"{len(feats)} frame | pose {summary[0]:.0%} | tay T {summary[1]:.0%} | tay P {summary[2]:.0%}",
                70, (0, 255, 0) if issue is None else (0, 0, 255))
            put(disp, "ENTER: luu | R: quay lai | F: luu du chat luong thap", 100)
            if issue:
                put(disp, ascii_text(issue), 130, (0, 0, 255))
        if auto and state not in ("PAUSED", "INTRO"):
            put(disp, "SPACE: tam dung | U: xoa mau vua luu | N: lop ke | Q: thoat", DISPLAY_SIZE - 50, scale=0.55)
        if msg:
            put(disp, msg, DISPLAY_SIZE - 18, (0, 200, 255), 0.6)
        cv2.imshow(WINDOW, disp)
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            return "quit"
        if key == ord("n"):
            return "next"
        if key == ord("u") and saved and state not in ("RECORDING", "REVIEW"):
            handler.remove_sample(name, saved.pop())
            total = handler.get_class_counts().get(name, 0)
            log("Đã xoá mẫu vừa lưu")
            if auto and state != "PAUSED":
                go("ARMING")
        elif auto and key == KEY_SPACE:
            if state == "PAUSED":
                rejects = 0
                go("ARMING")
                log("Tiếp tục")
            else:
                if state == "RECORDING":
                    log("Đã huỷ mẫu đang quay")
                go("PAUSED")
        elif state == "READY" and key == KEY_SPACE:
            go("COUNTDOWN")
            msg = ""
        elif state == "REVIEW":
            if key == ord("r"):
                go("READY")
                log("Đã bỏ mẫu vừa quay")
            elif key in (*KEY_ENTER, ord("f")):
                if key != ord("f") and issue is not None:
                    log(f"Chất lượng thấp ({issue}): bấm F để vẫn lưu, hoặc R để quay lại")
                    continue
                save()
                go("READY")
    return "done"

## Chạy thu thập
Chạy ô dưới rồi đứng vào vị trí. Ở chế độ `auto` chỉ cần hạ tay để bắt đầu mỗi mẫu. Muốn dùng phím thì bấm vào cửa sổ OpenCV trước để nó nhận phím.

In [5]:
cap = open_camera(CAMERA_INDEX, CAMERA_RESOLUTION)
det = Detector()
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
t_start = time.time()

try:
    for class_name, n in CLASS_PLAN.items():
        print(f"\n=== {class_name}: quay thêm {n} mẫu ===")
        if collect_class(cap, det, class_name, n) == "quit":
            break
finally:
    cap.release()
    det.close()
    cv2.destroyAllWindows()
    for _ in range(4):
        cv2.waitKey(1)

print(f"\nThời gian: {(time.time() - t_start) / 60:.1f} phút")
print_table()

Camera 0: 1920x1080 @ 30 fps (mặc định 1920x1080, backend AVFOUNDATION)


I0000 00:00:1795708895.364426  917508 init-domain.cc:134] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795708895.437648  917513 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795708895.454812  917513 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795708895.474256  917520 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795708895.487335  917520 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



=== gặp: quay thêm 20 mẫu ===


W0000 00:00:1795708896.053799  917516 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
W0000 00:00:1795708896.053799  917511 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Đã huỷ mẫu đang quay
Tiếp tục
Đã lưu gặp/0 (1/20, 31 frame, 15.2 fps)
Đã lưu gặp/1 (2/20, 31 frame, 15.5 fps)
Đã lưu gặp/2 (3/20, 30 frame, 15.0 fps)
Đã lưu gặp/3 (4/20, 31 frame, 15.3 fps)
Đã lưu gặp/4 (5/20, 31 frame, 15.5 fps)
Đã lưu gặp/5 (6/20, 31 frame, 15.5 fps)
Đã lưu gặp/6 (7/20, 31 frame, 15.3 fps)
Đã lưu gặp/7 (8/20, 31 frame, 15.4 fps)
Đã lưu gặp/8 (9/20, 31 frame, 15.4 fps)
Đã lưu gặp/9 (10/20, 31 frame, 15.4 fps)
Đã lưu gặp/10 (11/20, 31 frame, 15.5 fps)
Đã lưu gặp/11 (12/20, 31 frame, 15.2 fps)
Đã lưu gặp/12 (13/20, 31 frame, 15.2 fps)
Đã lưu gặp/13 (14/20, 31 frame, 15.4 fps)
Đã lưu gặp/14 (15/20, 31 frame, 15.4 fps)
Đã lưu gặp/15 (16/20, 31 frame, 15.5 fps)
Đã lưu gặp/16 (17/20, 31 frame, 15.2 fps)
Đã lưu gặp/17 (18/20, 30 frame, 15.0 fps)
Đã lưu gặp/18 (19/20, 31 frame, 15.2 fps)
Đã lưu gặp/19 (20/20, 31 frame, 15.2 fps)

Thời gian: 1.7 phút
Lớp                           Số mẫu
------------------------------------
a                                 45
age              

## Xem lại & dọn dẹp
Ô đầu liệt kê các mẫu kém nhất của session này. Ô thứ hai (đặt `REVIEW = True`) phát lại từng clip vừa quay để duyệt cả loạt: `ENTER`/`SPACE` giữ, `X` xoá, `B` quay lại clip trước, `Q` dừng.

In [6]:
rows = [r for r in handler.quality_report() if r["session"] == SESSION]
print(f"{len(rows)} mẫu trong session {SESSION}")
for r in sorted(rows, key=lambda r: r["pose_coverage"])[:10]:
    print(f"  {r['class']}/{r['sample_id']:<4} frames={r['frames']:<3} pose={r['pose_coverage']:.0%} "
          f"tayT={r['left_hand']:.0%} tayP={r['right_hand']:.0%}")

20 mẫu trong session 20261126-230129
  gặp/0    frames=31  pose=100% tayT=65% tayP=65%
  gặp/1    frames=31  pose=100% tayT=67% tayP=76%
  gặp/2    frames=30  pose=100% tayT=59% tayP=67%
  gặp/3    frames=31  pose=100% tayT=93% tayP=92%
  gặp/4    frames=31  pose=100% tayT=82% tayP=79%
  gặp/5    frames=31  pose=100% tayT=49% tayP=46%
  gặp/6    frames=31  pose=100% tayT=51% tayP=67%
  gặp/7    frames=31  pose=100% tayT=37% tayP=46%
  gặp/8    frames=31  pose=100% tayT=29% tayP=43%
  gặp/9    frames=31  pose=100% tayT=41% tayP=50%


In [ ]:
REVIEW = False

def play_clip(path, title):
    """Phát lặp clip tới khi có phím quyết định. Trả "keep" | "delete" | "back" | "quit"."""
    vid = cv2.VideoCapture(str(path))
    fps = vid.get(cv2.CAP_PROP_FPS) or 15.0
    clip = []
    while True:
        ok, fr = vid.read()
        if not ok:
            break
        clip.append(cv2.resize(fr, (DISPLAY_SIZE, DISPLAY_SIZE), interpolation=cv2.INTER_AREA))
    vid.release()
    if not clip:
        return "keep"
    while True:
        for fr in clip:
            disp = cv2.flip(fr, 1) if MIRROR_PREVIEW else fr.copy()
            put(disp, title, 36, (255, 255, 0), 0.8)
            put(disp, "ENTER/SPACE: giu | X: xoa | B: clip truoc | Q: dung", DISPLAY_SIZE - 18, scale=0.6)
            cv2.imshow(WINDOW, disp)
            key = cv2.waitKey(max(1, int(1000 / fps))) & 0xFF
            if key in (*KEY_ENTER, KEY_SPACE):
                return "keep"
            if key == ord("x"):
                return "delete"
            if key == ord("b"):
                return "back"
            if key == ord("q"):
                return "quit"

if REVIEW:
    clips = sorted(((r["class"], r["sample_id"]) for r in handler.quality_report() if r["session"] == SESSION),
                   key=lambda c: (c[0], int(c[1])))
    n_clips, n_deleted, i = len(clips), 0, 0
    cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
    try:
        while i < len(clips):
            cls_name, sid = clips[i]
            path = handler._class_dir(handler.raw, cls_name) / f"{sid}.mp4"
            action = play_clip(path, f"{ascii_text(cls_name)}/{sid}  ({i + 1}/{len(clips)})")
            if action == "quit":
                break
            if action == "delete":
                handler.remove_sample(cls_name, int(sid))
                clips.pop(i)
                n_deleted += 1
                print(f"Đã xoá {cls_name}/{sid}")
            else:
                i = max(0, i - 1) if action == "back" else i + 1
    finally:
        cv2.destroyAllWindows()
        for _ in range(4):
            cv2.waitKey(1)
    print(f"Đã xoá {n_deleted} / {n_clips} clip")
    print_table()

: 